# **Comparative Analysis of Literary Translations in the Odyssey: Type-Token Ratio in Depth**

This is a comparative Type-Token Ratio (TTR) analysis of six translations spanning roughly the last century: 1919–2018. 

## **Road Map**

### I. Libraries, files, and paths

### II. The Texts

1. Bibliographic information about the translators  
    a) Name, date, publisher, digital source  
    c) Etymologic bias, character names and place names  

2. The translators at a glance tokenwise  
    a) Lenght and distribution  
        - Total token-words ratio  
        - Book distribution  
        - Choosing a baseline: Murray(?)  
    b) Token quality check  
        - DataFrame of firstlines (books, authors)  
        - Visial auditing Book 6  
        - Purity index   
            * by book  
            * aggregated  
    c) 

### III. TTR Analysis

1. All-in, straightforward model  
    a) TTR Computation  
    b) Shapiro-Wilk test to check for normality  
    c) One-wat ANOVA for overall differences  
    d) Pairwise t-test using Bonferroni coprrection  
    c) Meassuring effect size ussing Cohen's d  

2. Adaptive models  
    a) Mixed-Effects model: author fixed effect / book as random effect  
    b) Standardized TTR:   
    c) Moving-average TTR: translation as temporal change  

3. Supplement models  
    a) Lexical Density   
    b) Diachronic analysis  
    c) Semantic fields:  

### IV. Discussing Results
    

## **I. Libraries, files, and paths**

In [1]:
#### Other libraries
import sys
import os

import ast
from collections import Counter

import re
import nltk

In [2]:
#### Pandas set-up
import numpy as np
import pandas as pd

In [3]:
#### Visualization
sys.path.append('/Users/debr/English-Homer') 
import matplotlib.pyplot as plt
import seaborn as sns
import bard_visualization as viz # My Vizualization library

Bard viz is on the stage!


In [4]:
# Import my functions
sys.path.append('/Users/debr/English-Homer/functions') 
import e_nlp as e

e enters! »----> use e.<function>

e.g., for pipeline
:	 nlp = e.NLPPipeline(language="english")
Stopwords customized:
  Added: {'eight', 'seven', 'nine', 'two', 'ten', "'and", 'six', 'n', 'one', 'five', 'four', 'three', "'"}
  Removed: {''}
  Total stopwords: 211
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'’', '\\', '‘', '—', '-', '”', '…', '“'}
  Punctuation to be removed: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~—‘’“”…


[nltk_data] Downloading package punkt_tab to /Users/debr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
#### Cell display options
pd.set_option("display.max_colwidth", None)  # Prevent truncation of long values
pd.set_option("display.max_rows", None)  # Show all rows
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.expand_frame_repr", False)  # Prevent wrapping in DataFrames

In [ ]:
#### File management
# TO UPDATE
nb_id = "Six_Modern_TTR"
odysseys = ["AT_Murray", "Fitzgerald", "Lattimore", "Fagles", "Wilson", "Green"]

# Paths
output_path = f"/Users/debr/English-Homer/{nb_id}/"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

output_path_plots = f"output_path/{nb_id}_plots/"
os.makedirs(os.path.dirname(output_path_plots), exist_ok=True)

In [ ]:
# Daframe with all the odysseys
dfs = []

for odyssey in odysseys:
    filepath = f"/Users/debr/odysseys_en/dataframed/Odyssey_{odyssey}_DataFrame.csv"
    temp_df = pd.read_csv(filepath)  
    dfs.append(temp_df)  # Append it to the list

df = pd.concat(dfs, axis=0, ignore_index=True)

df["text"] = df["text"].apply(ast.literal_eval)
df = df[['author', 'book_num', 'text', 'tokens', 'num_words', 'num_tokens']]
df["diff"] = df["num_words"] - df["num_tokens"]
e.check_df(df)

## **II. The Texts**
### **1. Bibliographic information about the translators**

Here is a brief description of each translation, starting with bibliographic data: a) Name, date, publisher, source, etc.; followed by labelling their b) Etymologic bias, regarding characters and places names.

| Author                  | Year | Period       | Span  | Verse | Publisher               | Edition | Source                | URL                                                                 | Notes | Bibliography |
|-------------------------|------|-------------|-------|-------|-------------------------|---------|------------------------|---------------------------------------------------------------------|-------|--------------|
| Augustus Taber Murray  | 1919 | Modernism   | 1910-1940 | FALSE | Harvard UP, Loeb          | 1919    | Scaife Viewer, Tufts University | [Link](https://scaife.perseus.org/library/urn:cts:greekLit:tlg0012.tlg002/) | "No more faithful translation of Homer was ever made, and its elegance matched its fidelity. Homer's formulaic epithets, phrases, and sentences were consistently rendered, and his artificial amalgam of dialects and archaic vocabulary were, as was perfectly acceptable in those days, reflected in archaic English." From the Preface to the revised 1995 edition, also in the Loeb collection.  |*Homer. The Odyssey, Vol. 1-2. Murray, A. T. (Augustus Taber), translator. London: William Heinmann; New York: G.P. Putnam's Sons, 1919.* |
| Robert Fitzgerald      | 1961 | (Ur)Postmodernism   | 1960-1979 | TRUE  | Doubleday & Company, Inc.  | Second 1963 Edition    | Internet Archive       | [Link](https://archive.org/details/homer-the-odyssey-800-bce-robert-fitzgerald-tr-1961) | "Written in very readable, American “plain style” language" (Carolyn Clark, Best American Poetry, 2020) | *Homer. The Odyssey. Fitzgerald, Robert, translator. New York: Doubleday & Company, Inc., 1961* | 
| Richmond Lattimore     | 1965 | (Ur)Postmodernism   | 1960-1979 | TRUE      |     Farrar, Straus & Giroux                    | First 1965 Edition          | Harper Collins E-Books               | NONE | "Prose-like verse translation" (Carolyn Clark, Best American Poetry, 2020). | *Homer. The Odyssey. Lattimore, Richmond, translator. New York: Doubleday & Company, Inc., 1965* | 
| Robert Fagles         | 1996 | Postmodernism | 1980-1999 | TRUE      |  Penguin                       | First 1996 Edition        | PDF SCAN               | NONE | | *Homer. The Odyssey. Fagles, Robert, translator. University of California Press, 1996*|
| Emily Wilson          | 2017 | Contemporary | >2000 | TRUE      | W.W. Norton & Company Inc.         | First 2017 Edition | PDF SCAN  | NONE| | *Homer. The Odyssey. Wilson, Emily, translator. W.W. Norton & Company Inc., 2017* |
| Peter Green           | 2018 | Contemporary | >2000 | TRUE | University of California Press | Digital | EBSCOhost | [Link](https://search.ebscohost.com/login.aspx?direct=true&scope=site&db=nlebk&db=nlabk&AN=1696363) | | *Homer. The Odyssey. Green, Peter, translator. University of California Press, 2018* |

Use of Latin vs Greek form for characters' name by translator. There is an inconsistency in their us. Up until Fagles, every translator used a mix of direct Greek transliteratio and the conventional name passed on by the earlier tradition. 

- **Murray:** Lean to use Greek (Telemakhos, Menelaos) yet more popular names (Odysseus, Athena) reamin in their Latin form
- **Fitzgerald:** Opts for transliterating Greek phonetics, like Murray Telemakhos but to an extreme in Akhilleus, however others retain their popular form.
- **Lattimore:** Less Greek than Fitzgerald (Telemachus) but still Athene and Achilleus
- **Fagles:** Experiment less with character names and sttles on the convention
- **Wilson:** Follows the same names as Fagles
- **Green:** Returns to a Greek flavor by including Greek diacritics to the names: Achillēs, Menelaös, Tēlemachos, Athēnē

The names of places follows the same patter. This will be analyzed thoroughly in the Name Entity Recognition step of the project.

#### 2. **The translators at a glance tokenwise**
**a) Lenght and distribution**  
- Total token-words ratio  
- Book distribution  
- Choosing a baseline: Murray(?)  

In [ ]:
# Create a color mapping for authors
authors = df["author"].unique()
palette = {author: color for author, 
           color in zip(authors, viz.color_palette.values())}
palette

In [ ]:
# Compute diff
df["diff"] = df["num_words"] - df["num_tokens"]

# Aggregate sum per author
agg_diff = df.groupby("author")["diff"].sum()
mean_agg_diff = agg_diff.mean()



# Plot
sns.barplot(data=df, x="author", y="diff", estimator=sum, errorbar=None,
            hue="author", palette=palette)  # Explicit color mapping

# Mean horizontal line
plt.axhline(mean_agg_diff, color="red", linestyle="--",
            label=f"Mean: {mean_agg_diff:.2f}")

# Labels and title
plt.xlabel("Author")
plt.ylabel("Sum of (num_words - num_tokens)")
plt.title("Difference Between num_words and num_tokens by Translator")
plt.legend()
plt.xticks(rotation=45)  # Rotate x-axis labels for better readability

# Show plot
plt.show()

In [ ]:
# Side by side comparison of num_words and num_tokens by translator
# Reshape the DataFrame to a long format
df_melted = df.melt(id_vars=["author"], value_vars=["num_words", "num_tokens"],
                     var_name="Metric", value_name="Count")

# Create the bar plot
plt.figure(figsize=(12, 6))
sns.barplot(data=df_melted, x="author", y="Count", hue="Metric", 
            estimator=sum, errorbar=None)

# Customize labels
plt.xticks(rotation=0)
plt.xlabel("Author")
plt.ylabel("Count")
plt.title("Comparison of num_words and num_tokens by Author")
plt.legend(title="Metric")
#plt.savefig(output_path_plots + f"words_tokens-{nb_id}.png")
plt.show()

In [ ]:
df["token_word_ratio"] = df["num_tokens"] / df["num_words"]
# Print token word ratio by author
print(df.groupby("author")["token_word_ratio"].mean())

In [ ]:
baseline_author = "Fagles"  # Change to the author you want
baseline_ratio = df[df["author"] == baseline_author]["token_word_ratio"].mean()

In [ ]:
# Compute mean token-word ratio per author
author_means = df.groupby("author")["token_word_ratio"].mean().reset_index()

# Bar plot
sns.barplot(data=author_means, x="author", y="token_word_ratio", hue="author", palette=palette)

# Baseline (horizontal line for reference author)
plt.axhline(baseline_ratio, color="red", linestyle="--", label=f"{baseline_author} Mean: {baseline_ratio:.2f}")

# Labels and title
plt.xlabel("Author")
plt.ylabel("Token-Word Ratio")
plt.title("Token-Word Ratio by Translator")
plt.xticks(rotation=45)  # Rotate labels for better readability
plt.legend()

# Show plot
plt.show()

####    b) Token quality check

- DataFrame of firstlines (books, authors)
- Visial auditing of the Proems by author
- Purity index 
    * by book
    * aggregated

In [ ]:
# DF of First lines

def extract_first_lines(df):
    """
    Extracts the first line of each book and groups by author.

    Args:
        df (pd.DataFrame): DataFrame with columns ['author', 'book_num', 'text'].

    Returns:
        pd.DataFrame: New DataFrame with ['author', 'book_num', 'first_line'].
    """
    # Ensure the necessary columns exist
    if not all(col in df.columns for col in ["author", "book_num", "text"]):
        raise ValueError("Missing required columns: ['author', 'book_num', 'text']")
    
    # Extract first line of each book
    df["first_line"] = df["text"].apply(lambda x: x.split("\n")[0] if isinstance(x, str) and "\n" in x else x)

    # Select and return relevant columns
    return df[["author", "book_num", "first_line"]].sort_values(by=["author", "book_num"])

# Example usage:
new_df = extract_first_lines(df)
new_df

In [ ]:
df["text"][0][:3] 

In [ ]:
def extract_first_lines(df):
    """
    Extracts the first 9 lines of Book 1 for each author.
    
    Args:
        df (pd.DataFrame): The input DataFrame containing columns 'author', 'book_num', and 'text'.
    
    Returns:
        pd.DataFrame: A new DataFrame with authors and their first 9 lines from Book 1.
    """
    first_lines_data = []

    for author in df["author"].unique():  # Loop through each unique author
        book1_text = df.loc[(df["author"] == author) & (df["book_num"] == 1), "text"].values

        if len(book1_text) > 0:  # Ensure there's data
            first_9_lines = "".join(book1_text[0][:9])  # Get first 9 lines
            first_lines_data.append({"author": author, "proem": first_9_lines})

    # Convert the list to a DataFrame
    df_proem = pd.DataFrame(first_lines_data)
    return df_proem


df_proem = extract_first_lines(df)
df_proem

____________________________________________________
## 📖  Experiment 1: **Type-Token Ratio (TTR)** 

## ✨ Introduction  

**Type-Token Ratio (TTR)** is a key measure of lexical diversity, calculated as the number of unique words (**types**) divided by the total word count (**tokens**). It provides insight into a translator’s lexical choices when rendering the same source text.  

- A **higher TTR** suggests a richer vocabulary, possibly reflecting an effort to capture nuances or stylistic complexity.  
- A **lower TTR** may indicate a more repetitive or constrained word choice, potentially prioritizing accessibility or fidelity to the original.  

### 📌 **TTR Formula**  

$$TTR = \left( \frac{\text{Unique Words}}{\text{Total Words}} \right) \times 100$$  


In [ ]:
import scipy.stats as stats

# Compute TTR
df["ttr"] = df["tokens"].apply(lambda x: (len(set(x)) / len(x) * 100) if x else 0)


In [ ]:
# List of translators
translators = ["AT_Murray", "Fitzgerald", "Lattimore", "Fagles", "Wilson", "Green"]
ttr_by_translator = {}
for translator in translators:
    ttr_by_translator[translator] = df[df["author"] == translator]["ttr"].tolist()

# TTR DF by translator
if 'book_num' not in df.columns:
    df['book_num'] = [f"Book_num_{i+1}" for i in range(1, 25)] * len(translators)
ttr_df = df.pivot(index='book_num', columns='author', values='ttr')

# Reorder columns if needed
ttr_df = ttr_df[translators]
e.check_df(ttr_df)

In [ ]:
from scipy import stats

for translator in translators:
    # Get the TTR data for this translator
    ttr_data = ttr_by_translator[translator]
    
    # Perform Shapiro-Wilk test
    stat, p_value = stats.shapiro(ttr_data)
    
    # Print results
    print(f"Shapiro-Wilk test for {translator}'s data: T-statistic={stat:.4f}, p-value={p_value:.4f}")
    
    # Interpret results
    if p_value < 0.05:
        print(f"{translator}'s TTR data is not normally distributed.")
    else:
        print(f"{translator}'s TTR data is normally distributed.")
    
    print() # Add empty line for readability

In [ ]:
import scipy.stats as stats
import numpy as np
from itertools import combinations

# Use the ttr_by_translator dictionary
translator_names = translators  # Use your existing list of translators
ttr_values = [ttr_by_translator[translator] for translator in translator_names]

# Perform one-way ANOVA
f_stat, p_value = stats.f_oneway(*ttr_values)
print(f"F-statistic: {f_stat:.4f}, P-value: {p_value:.4f}")

if p_value < 0.05:
    print("There are statistically significant differences in TTR among the translators.")
else:
    print("There are no statistically significant differences in TTR among the translators.")

# Perform pairwise t-tests with Bonferroni correction
print("\nPairwise comparisons:")

# Number of comparisons for Bonferroni correction
num_comparisons = len(list(combinations(range(len(translator_names)), 2)))

for i, j in combinations(range(len(translator_names)), 2):
    t_stat, p_val = stats.ttest_ind(ttr_values[i], ttr_values[j])
    
    # Apply Bonferroni correction
    adj_p_val = min(p_val * num_comparisons, 1.0)
    
    # Calculate mean difference
    mean_diff = np.mean(ttr_values[i]) - np.mean(ttr_values[j])
    
    # Determine significance
    is_significant = "Significant" if adj_p_val < 0.05 else "Not significant"
    
    print(f"{translator_names[i]} vs {translator_names[j]}: Diff = {mean_diff:.4f}, p = {adj_p_val:.4f} - {is_significant}")


In [ ]:
# Perform t-test
t_stat, p_value = stats.ttest_ind(ttr_by_translator['Wilson'], ttr_by_translator['Green'])
print(f"T-statistic: {t_stat}, P-value: {p_value}")
if p_value < 0.05:
    print("The difference in TTR between Wilson and Green is statistically significant.")
else:
    print("The difference in TTR between Wilson and Green is not statistically significant.")

In [ ]:
import numpy as np
from scipy import stats

def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    mean_diff = np.mean(group1) - np.mean(group2)
    return mean_diff / pooled_std

# Calculate Cohen's d for significant differences
significant_pairs = [
    ('Lattimore', 'Wilson'),
    ('Fagles', 'Wilson'),
    ('Wilson', 'Green'),
    ('Fitzgerald', 'Wilson'),
    ('Fitzgerald', 'Green'),
    ('Fitzgerald', 'Lattimore'),
    ('Fitzgerald', 'Fagles'),
    ('Lattimore', 'Fagles'),
    ('Lattimore', 'Green'),
    ('Fagles', 'Green')
]

for pair in significant_pairs:
    translator1, translator2 = pair
    group1 = ttr_by_translator[translator1]
    group2 = ttr_by_translator[translator2]
    
    d = cohens_d(group1, group2)
    t_stat, p_value = stats.ttest_ind(group1, group2)
    
    print(f"{translator1} vs {translator2}:")
    print(f"Cohen's d: {d:.4f}")
    print(f"T-statistic: {t_stat:.4f}")
    print(f"P-value: {p_value:.4e}")
    
    # Interpret effect size
    if abs(d) < 0.2:
        interpretation = "small"
    elif abs(d) < 0.5:
        interpretation = "medium"
    else:
        interpretation = "large"
    
    print(f"Effect size interpretation: {interpretation}")
    print()


In [ ]:
def additional_metrics(group1, group2):
    u3 = stats.norm.cdf(d).mean() * 100  # Percentage above average
    overlap = 2 * stats.norm.cdf(-abs(d)/2) * 100  # Visual overlap
    return u3, overlap
additional_metrics(group1, group2)

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Assuming your DataFrame is named 'df' and has columns 'author', 'book_num', and 'ttr'
# If not, adjust the column names accordingly

# Fit the mixed-effects model
# 'ttr' is the dependent variable
# 'author' is the fixed effect (the translators you want to compare)
# 'book_num' is the random effect (to account for book-to-book variation)

# The formula specifies the model
# C(author) tells the model that 'author' is a categorical variable
# The random intercept is specified as '1 | book_num'
model = smf.mixedlm("ttr ~ C(author)", data=df, groups=df["book_num"])

# Fit the model
results = model.fit()

# Print the results summary
print(results.summary())

# If you want to perform post-hoc tests, you can use the 'pairwise_tukeyhsd' function
# from the statsmodels.stats.multicomp module. However, this function is not
# directly compatible with the mixed-effects model results. You would need to
# extract the estimated marginal means (least squares means) and their standard errors
# from the mixed-effects model results and then use those values in the 'pairwise_tukeyhsd'
# function.

# The results summary will give you the estimated coefficients for each translator,
# as well as standard errors, t-values, and p-values. You can use these values
# to compare the translators and determine whether there are significant differences
# in TTR between them.


In [ ]:
print(df.sort_values('author').head())

In [ ]:
import pandas as pd
import numpy as np

# Assuming you have a DataFrame 'df' with columns 'author', 'book_num', 'text', and 'tokens'

# ----------------------------------------------------------------------
# Standardized TTR (STTR)
# ----------------------------------------------------------------------

def calculate_sttr(tokens, segment_size=100):
    """
    Calculates Standardized Type-Token Ratio (STTR).

    Args:
        tokens (list): List of tokens (words).
        segment_size (int): Size of each segment to calculate TTR on.

    Returns:
        float: STTR value.
    """
    if len(tokens) < segment_size:
        return len(set(tokens)) / len(tokens) * 100  # Fallback to regular TTR if too short

    num_segments = len(tokens) // segment_size
    ttr_values = []

    for i in range(num_segments):
        segment = tokens[i * segment_size: (i + 1) * segment_size]
        ttr = len(set(segment)) / len(segment) * 100
        ttr_values.append(ttr)

    return np.mean(ttr_values)

# Apply STTR calculation to your DataFrame
df['sttr'] = df['tokens'].apply(calculate_sttr)

# ----------------------------------------------------------------------
# Moving-Average TTR
# ----------------------------------------------------------------------

def calculate_moving_average_ttr(tokens, window_size=100):
    """
    Calculates Moving-Average Type-Token Ratio.

    Args:
        tokens (list): List of tokens (words).
        window_size (int): Size of the moving window.

    Returns:
        list: List of moving-average TTR values.
    """
    if len(tokens) < window_size:
        return [len(set(tokens)) / len(tokens) * 100]  # Fallback to regular TTR if too short

    moving_average_ttr = []
    for i in range(window_size, len(tokens) + 1):
        window = tokens[i - window_size:i]
        ttr = len(set(window)) / len(window) * 100
        moving_average_ttr.append(ttr)
    return moving_average_ttr

# Apply Moving-Average TTR calculation to your DataFrame
# This creates a new column with lists of TTR values for each book
df['moving_average_ttr'] = df['tokens'].apply(calculate_moving_average_ttr)


# ----------------------------------------------------------------------
# Example Usage and Analysis
# ----------------------------------------------------------------------

# Now you can analyze the STTR and moving_average_ttr
# For example, calculate the average STTR for each translator:
average_sttr_by_translator = df.groupby('author')['sttr'].mean()
print("Average STTR by translator:\n", average_sttr_by_translator)

# For moving average TTR, you might want to plot the TTR values over the course of each book

# Example: Print the first few moving average TTR values for the first book of AT_Murray
first_book_at_murray = df[(df['author'] == 'AT_Murray') & (df['book_num'] == 1)]['moving_average_ttr'].iloc[0] #Adapt book_num
print("\nMoving Average TTR for first book of AT_Murray (first few values):\n", first_book_at_murray[:5])


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  # Import seaborn

# Assuming you have a DataFrame 'df' with columns 'author', 'book_num', 'text', and 'tokens'
# and that you've already calculated the moving_average_ttr column as in the previous response

# ----------------------------------------------------------------------
# Plotting Moving-Average TTR
# ----------------------------------------------------------------------

def plot_moving_average_ttr(df, author, book_num, ax=None):
    """
    Plots the moving average TTR for a specific author and book.

    Args:
        df (pd.DataFrame): DataFrame containing the data.
        author (str): Name of the author.
        book_num (int): Book number.
        ax (matplotlib.axes._axes.Axes, optional): Matplotlib axes object. If None, creates a new figure.
    """
    data = df[(df['author'] == author) & (df['book_num'] == book_num)]['moving_average_ttr'].iloc[0]

    if not data:
        print(f"No data found for author {author}, book {book_num}")
        return

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(data)
    ax.set_title(f"Moving Average TTR - {author}, Book {book_num}")
    ax.set_xlabel("Window Position")
    ax.set_ylabel("TTR")
    return ax # Return the axes object for potential further customization

# Example Usage:
# Plot moving average TTR for each book by each author in subplots

authors = df['author'].unique()
book_nums = df['book_num'].unique()

num_authors = len(authors)
num_books = len(book_nums)

fig, axes = plt.subplots(num_authors, num_books, figsize=(15, 5*num_authors), sharex=True, sharey=True) #Adjusted figure size and sharing axes

# Flatten the axes array for easier indexing
axes = axes.flatten()

for i, author in enumerate(authors):
    for j, book_num in enumerate(book_nums):
        ax_index = i * num_books + j  # Calculate the index for the subplot
        ax = axes[ax_index]  # Get the axes object
        plot_moving_average_ttr(df, author, book_num, ax=ax)

        # Set x and y labels only for the first row and column to avoid repetition
        if i == num_authors - 1:
            ax.set_xlabel("Window Position")  # X label only on the bottom row
        if j == 0:
            ax.set_ylabel("TTR")  # Y label only on the first column

        # Set title only for the first row
        if i == 0:
            ax.set_title(f"Book {book_num}")  # Title on the top row only

        ax.set_title(f"{author}, Book {book_num}", fontsize=8)  # Consistent title for each subplot

# Adjust layout to prevent overlapping titles/labels
plt.tight_layout(rect=[0, 0.03, 1, 0.97])  # Add space for overall figure title
plt.suptitle("Moving Average TTR by Author and Book", fontsize=16)  # Overall figure title

# Show the plot
plt.show()


In [ ]:
e.check_df(df)